the derivative (ableitung) tells you how sensitive a function `f` is to changes in `x`.

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

numerically we can measure this by nudging `x` by a small amount of `h` and dividing the resulting change in `f` by `h`.
(as `h` can only get close to but never be `0`, numerically we can only get an approximation.)

---

the value we get is called the slope (steigung) – a factor of sensitivity

as `f` is most likely non-linear, the slope only holds true in a tiny neighbourhood. if you nudge a too big amount
before recomputing the slope, you may overshoot the desired minimum (assuming that `f` is a loss function which we want
to minimize)

---

before, we approximated the factor by which `f` changes in relation to `x` which is expressed as

$$\frac{\Delta f}{\Delta x}$$

instead of a delta, the exact factor is written as `d`

$$f'(x) = \frac{df}{dx}$$

---

if we take an example of a loss function `L(a,b,c)` we now have multiple parameters, each with their own slope and
effect on `L`.

therefore the `d` notation above would be imprecise if we are just looking at `b` e.g., as other parameters also have
influence. this is expressed as a partial derivative

$$\frac{\partial L}{\partial b} = \text{slope of } L \text{ with respect to } b$$

---

in the context of neural networks, we want to look at multiple parameters at the same time.

for each parameter, its partial derivative tells us how `L` reacts to that one parameter. all of them combined into one
vector is the gradient of `L`

$$\nabla L = \left[\frac{\partial L}{\partial a},\frac{\partial L}{\partial b},\frac{\partial L}{\partial c}\right]$$

---

now we want to start moving step by step into a direction that minimizes our loss function

for each value in the gradient vector, its sign tells us which way to move and its magnitude how far. in the case of
`0` we do not move, as this parameter has no influence on `L`

we think of the gradient as a vector pointing in the direction of increased loss.

multiplying a partial derivative by a learning rate `lr` scales the step.

$$\text{lr} \cdot \frac{\partial L}{\partial p}$$

training of a neural net moves in the opposite direction, as we aim to decrease the loss function, which looks as
follows for the parameter `p`

$$p \leftarrow p - \text{lr} \cdot \frac{\partial L}{\partial p}$$

---

in practice, we never measure anything using h.

for a simple MLP with `w*x+b`, we only need three operators/functions: `+`, `*`, and our activation function of choice,
in this case `tanh`. for each we know their respective local derivative rule.

local derivative means the derivative of an operation/function, not a partial derivative of `L`!

an autograd implementation records every operation of a function’s execution (forward pass) as a graph.

the magic of backpropagation is that you can now walk this graph backwards, and apply the *chain rule* to get the
partial derivative `∂L/∂p` per parameter `p`.

concretely, each operation/function, during `backward()`, multiplies its local derivative by the gradient coming in from
its parent (when looking from `L` backwards), then adds that into the child's gradient *(`+=`, not `=`)*.

`+=` because a value can be used more than once in the graph (e.g. `b = a + a`) - each use adds its own influence, they
shouldn't overwrite each other. that's also why every `.grad` starts at `0`.

---

![neuron_model](imgs/neuron_model.jpeg)

---

the bias gives the neuron its missing degree of freedom.

`y = w*x + b` can be any line, not just ones passing through the origin. without `b`, `y = w*x` is forced to go through
`(0,0)`

weights need randomness to break symmetry between neurons in a layer. otherwise they would all learn identically.

the bias doesn't have this problem, since that asymmetry is already given by the random weights. so `0` is a safe,
neutral starting point for the bias

a random bias would add unnecessary variance to the pre-activation → risk of early tanh saturation → smaller
initial gradients → slow to basically halting training

pre-activation is the value before it has passed through the activation function

---

when we think of neurons in a neural net, we don't primarily use activation functions like `tanh` to normalize the
output to a specific range – the main reason is non-linearity.

stacking multiple `w*x+b` layers without anything in between, they would collapse into a single linear function, since a
linear combination of linear functions is still linear. a non-linear function between layers is what lets the network
approximate more complex, non-linear relationships.

---

the last missing piece for a simple MLP (multi layer perceptron) is to define our loss function `L`.

three requirements:

1. the loss function must somehow use the predictions of the MLP in its calculation
2. the loss function calculation must fully use the autograd `Value` scalars

(otherwise there would be no complete autograd graph pointing from the result of `L` all the way backwards)

3. the loss function must return a single scalar `Value`

---

> what I think is interesting is that I would have expected a way tighter coupling between the loss function and the parameters

---

![neural_net](imgs/neural_net.jpeg)

---